<a href="https://colab.research.google.com/github/noname123920/diplom/blob/main/PINN_DeepXDE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install deepxde

## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation $p_t = \kappa p_{xx}$ for $x \in [0, \pi]$, $t \in [0, 1]$, with initial condition $p(x, 0) = \sin(x)$ and boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [2]:
import deepxde as dde
import numpy as np
from deepxde.backend import tf

Using backend: tensorflow
Other supported backends: tensorflow.compat.v1, pytorch, jax, paddle.
paddle supports more examples now and is recommended.


In [3]:
kappa = 1.0

In [4]:
geom = dde.geometry.Interval(0, np.pi)
timedomain = dde.geometry.TimeDomain(0, 1)
geomtime = dde.geometry.GeometryXTime(geom, timedomain)

In [5]:
def pde(x, p):
    dp_t = dde.grad.jacobian(p, x, i=0, j=1)
    dp_xx = dde.grad.hessian(p, x, i=0, j=0)
    return dp_t - kappa * dp_xx

In [6]:
def func(x):
    xx = x[:, 0]
    tt = x[:, 1]
    return (np.sin(xx) * np.exp(-tt))[:, None]

In [7]:
data = dde.data.TimePDE(
    geomtime,
    pde,
    [],
    num_domain=100,
    solution=func
)

In [8]:
layer_size = [2] + [20] * 2 + [1]
activation = "tanh"
initializer = "Glorot uniform"

In [9]:
net = dde.nn.FNN(layer_size, activation, initializer)

In [10]:
def output_transform(x, y):
    x_col = x[:, 0:1] # столбец x, форма (N, 1)
    t_col = x[:, 1:2] # столбец t, форма (N, 1)
    return tf.sin(x_col) + x_col * (np.pi - x_col) * t_col * y

net.apply_output_transform(output_transform)

In [11]:
model = dde.Model(data, net)
model.compile("adam", lr=1e-3, metrics=["l2 relative error"])

Compiling model...
'compile' took 0.031564 s



In [12]:
resampler = dde.callbacks.PDEPointResampler(period=1)

In [13]:
losshistory, train_state = model.train(iterations=1000, callbacks=[resampler], display_every=50)

Training model...

Step      Train loss    Test loss     Test metric   
0         [1.64e+00]    [1.64e+00]    [6.36e-01]    
50        [1.13e-01]    [1.13e-01]    [1.79e-01]    
100       [3.07e-02]    [3.07e-02]    [1.03e-01]    
150       [1.19e-02]    [1.19e-02]    [4.47e-02]    
200       [7.69e-03]    [7.69e-03]    [2.88e-02]    
250       [5.78e-03]    [5.78e-03]    [2.44e-02]    
300       [4.48e-03]    [4.48e-03]    [2.14e-02]    
350       [3.50e-03]    [3.50e-03]    [1.87e-02]    
400       [2.73e-03]    [2.73e-03]    [1.63e-02]    
450       [2.13e-03]    [2.13e-03]    [1.41e-02]    
500       [1.67e-03]    [1.67e-03]    [1.22e-02]    
550       [1.32e-03]    [1.32e-03]    [1.05e-02]    
600       [1.05e-03]    [1.05e-03]    [8.99e-03]    
650       [8.43e-04]    [8.43e-04]    [7.76e-03]    
700       [6.85e-04]    [6.85e-04]    [6.72e-03]    
750       [5.63e-04]    [5.63e-04]    [5.87e-03]    
800       [4.69e-04]    [4.69e-04]    [5.16e-03]    
850       [3.95e-04]    [3.

In [14]:
import plotly.express as px
import pandas as pd
import numpy as np

In [15]:
domain_x = [0, np.pi]
domain_t = [0, 1]

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
    x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
    t_plot = t_val * np.ones_like(x_plot)
    inputs_plot = np.concatenate([x_plot, t_plot], axis=1)

    p_plot = model.predict(inputs_plot)

    p_exact = np.exp(-t_val) * np.sin(x_plot)
    df_temp = pd.DataFrame({
        'x': x_plot.flatten(),
        'PINN': p_plot.flatten(),
        'Exact': p_exact.flatten(),
        'Time': t_val
    })
    plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')

fig = px.line(df_melted, x='x', y='p', animation_frame='Time',
              color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(xaxis_title='x', yaxis_title='p(x, t)')
fig.show()

## Loss:

In [16]:
import plotly.graph_objects as go
import numpy as np

In [17]:
losses = np.array(losshistory.loss_train).flatten()
steps = np.array(losshistory.steps)

fig = go.Figure()
fig.add_trace(go.Scatter(x=steps, y=losses, mode='lines+markers', name='Dirichlet'))
fig.update_layout(
    title='Сходимость обучения PINN (DeepXDE)',
    xaxis_title='Итерация',
    yaxis_title='Функция потерь',
    yaxis_type='log',
    template='plotly_white'
)
fig.show()

## Что будет если менять число эпох?

In [26]:
import random

In [27]:
checkpoints = [1000, 2000, 3000, 4000, 5000]
seeds = [1, 2, 3]
errors = {}

In [20]:
x = np.linspace(0, np.pi, 200)
t = np.linspace(0, 1, 100)
X, T = np.meshgrid(x, t)
inputs = np.stack([X.ravel(), T.ravel()], axis=1)
p_exact = np.exp(-T) * np.sin(X)

In [24]:
import tensorflow.compat.v2 as tf
import deepxde.nn.initializers as dde_initializers

dde_initializers.INITIALIZER_DICT["Glorot uniform"] = tf.keras.initializers.GlorotUniform(seed=101)

In [25]:
for step in checkpoints:
    l2_runs = []
    for seed in seeds:
        dde.config.set_random_seed(seed)
        dde_initializers.INITIALIZER_DICT["Glorot uniform"] = tf.keras.initializers.GlorotUniform(seed=seed)

        data_i = dde.data.TimePDE(geomtime, pde, [], num_domain=100, solution=func)

        net_i = dde.nn.FNN(layer_size, activation, initializer)
        net_i.apply_output_transform(output_transform)
        model_i = dde.Model(data_i, net_i)
        model_i.compile("adam", lr=1e-3)

        resampler_i = dde.callbacks.PDEPointResampler(period=1)
        model_i.train(iterations=step, callbacks=[resampler_i], display_every=step)

        p_pred = model_i.predict(inputs).reshape(X.shape)
        l2_runs.append(np.linalg.norm(p_pred - p_exact) / np.linalg.norm(p_exact))

    errors[step] = np.mean(l2_runs)
    print(f"{step} -> L2 = {errors[step]:.4e} (mean of {len(seeds)})")

Compiling model...
'compile' took 0.008153 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.29e-01]    [2.29e-01]    []  
1000      [2.52e-04]    [2.52e-04]    []  

Best model at step 1000:
  train loss: 2.52e-04
  test loss: 2.52e-04
  test metric: []

'train' took 14.273887 s

Compiling model...
'compile' took 0.012044 s

Training model...

Step      Train loss    Test loss     Test metric
0         [2.38e-01]    [2.38e-01]    []  
1000      [7.90e-05]    [7.90e-05]    []  

Best model at step 1000:
  train loss: 7.90e-05
  test loss: 7.90e-05
  test metric: []

'train' took 13.456513 s

Compiling model...
'compile' took 0.020994 s

Training model...

Step      Train loss    Test loss     Test metric
0         [9.37e+00]    [9.37e+00]    []  
1000      [2.53e-03]    [2.53e-03]    []  

Best model at step 1000:
  train loss: 2.53e-03
  test loss: 2.53e-03
  test metric: []

'train' took 13.393532 s

1000 -> L2 = 6.3585e-03 (mean of 3)
Compiling mod

In [28]:
checkpoints = [1000, 2000, 3000, 4000, 5000]
results = np.zeros((10, len(checkpoints)))